### **Imports**

In [1]:
import cv2
import numpy as np
from skimage import color, io
from skimage.metrics import structural_similarity as ssim
from skopt import gp_minimize
from skopt.space import Real, Integer
import pandas as pd
import os

**Utility Functions**

In [2]:
def load_images_from_folder(folder):
    images = [
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith('.png')
    ]
    return images

def save_information(results, param_csv):
    
    param_keys = ["alpha", "beta", "h", "hColor", "template", "search"]
    mean_params = {k: np.mean([r[k] for r in results]) for k in param_keys}
    
    os.makedirs(os.path.dirname(param_csv), exist_ok=True)
    pd.DataFrame([mean_params]).to_csv(param_csv, index=False)
    
    print(f"[INFO] Mean hyperparameters saved at: {param_csv}")
    return mean_params

### **Metrics**

**PSNR**

In [3]:
def calculate_psnr(imageA, imageB):
    # Ensure the images have the same size
    if imageA.shape != imageB.shape:
        raise ValueError("The images must have the same dimensions.")

    psnr_value = cv2.PSNR(imageA, imageB)

    return psnr_value


**SSIM**

In [4]:
def calculate_ssim(image1, image2, win_size=3):

    if image1 is None or image2 is None:
        raise ValueError("One or both images could not be loaded.")

    image1 = cv2.cvtColor(image1, cv2.COLOR_BGR2RGB)
    image2 = cv2.cvtColor(image2, cv2.COLOR_BGR2RGB)

    if image1.shape != image2.shape:
        raise ValueError("Both images must have the same dimensions.")

    ssim_value = ssim(image1, image2, multichannel=True, win_size=win_size)

    return ssim_value

**BRISQUE**

In [5]:
def calculate_mscn_coefficients(image):
    c = np.fft.fft2(image)
    c_shifted = np.fft.fftshift(c)
    magnitude = np.abs(c_shifted)
    log_magnitude = np.log(1.0 + magnitude)
    c_shifted_real = np.real(c_shifted)
    c_shifted_imag = np.imag(c_shifted)
    return c_shifted_real, c_shifted_imag, magnitude, log_magnitude

def calculate_mscn_features(image):
    _, _, _, log_magnitude = calculate_mscn_coefficients(image)
    std_dev = np.std(log_magnitude)
    mean = np.mean(log_magnitude)
    return [std_dev, mean]

def brisque_features(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    ms_std_dev, ms_mean = calculate_mscn_features(gray)
    return [ms_std_dev, ms_mean]

def getBRISQUE(image):
    features = brisque_features(image)
    weights = [-0.0977446, 0.0270277, 0.00090095, 0.0793246, 0.0476165, -0.033992, -0.0535509, 0.276186, 0.189205, 0.255546,
               0.120626, 0.0471861, -0.18469, 0.154051, -0.173411, -0.413456]
    intercept = 18.9217
    score = intercept
    for i in range(len(features)):
        score += features[i] * weights[i]
    return score

**NIQE**

In [6]:
def getNIQE(img):
    imagem_gray = color.rgb2gray(img)

    mean_local_contrast = np.mean(np.abs(np.gradient(imagem_gray)))
    std_local_contrast = np.std(np.abs(np.gradient(imagem_gray)))

    niqe_score = 1.0 / (1 + 6.6 * mean_local_contrast + 0.228 * std_local_contrast)

    return niqe_score

In [7]:
def apply_metrics(img_result, img_ref):

    psnr = calculate_psnr(img_result, img_ref)
    ssim_value = calculate_ssim(img_result, img_ref)
    brisque = getBRISQUE(img_result)
    niqe = getNIQE(img_result)

    return(psnr, ssim_value, brisque, niqe)

### **Image Processing**

In [8]:
def apply_filter(image, alpha, beta, h, hColor, templateSize, searchSize):

    adjusted = cv2.convertScaleAbs(image, alpha=alpha, beta=beta)
    filtered = cv2.fastNlMeansDenoisingColored(adjusted, None, h, hColor, templateSize, searchSize)
    return filtered

In [9]:
def objective_function(params, image, img_ref):
    alpha, beta, h, hColor, templateSize, searchSize = params
    filtered = apply_filter(image, alpha, beta, h, hColor, templateSize, searchSize)
    psnr, ssim_val, brisque, niqe = apply_metrics(filtered, img_ref)
    return -psnr - ssim_val + brisque + niqe

**Image Pair Optimization**

In [10]:
def optimize_image_pair(image_path, ref_path, space, n_calls=30):
    image = cv2.imread(image_path)
    img_ref = cv2.imread(ref_path)
    name = os.path.splitext(os.path.basename(image_path))[0]

    result = gp_minimize(lambda p: objective_function(p, image, img_ref), space,
                         n_calls=n_calls, random_state=0)

    best_alpha, best_beta, best_h, best_hColor, best_temp, best_search = result.x
    final_img = apply_filter(image, best_alpha, best_beta, best_h, best_hColor, best_temp, best_search)
    psnr, ssim_val, brisque, niqe = apply_metrics(final_img, img_ref)

    print(f"[{name}] alpha={best_alpha:.3f} | beta={best_beta:.3f} | "
          f"h={best_h:.3f} | hColor={best_hColor:.3f} | "
          f"template={best_temp:.3f} | search={best_search:.3f} | "
          f"PSNR={psnr:.3f} | SSIM={ssim_val:.3f} | BRISQUE={brisque:.3f} | NIQE={niqe:.3f}")

    return {
        "name": name,
        "alpha": best_alpha,
        "beta": best_beta,
        "h": best_h,
        "hColor": best_hColor,
        "template": best_temp,
        "search": best_search,
        "psnr": psnr,
        "ssim": ssim_val,
        "brisque": brisque,
        "niqe": niqe
    }

**Optimization Phase**

In [11]:
def optimization_phase(low_images, high_images, space):
    results = []
    for low_path, high_path in zip(low_images, high_images):
        result = optimize_image_pair(low_path, high_path, space)
        results.append(result)

    return results

In [12]:
def evaluation_phase(low_images, high_images, mean_params, out_folder, metrics_csv):
    os.makedirs(out_folder, exist_ok=True)
    metrics_list = []

    for low, high in zip(low_images, high_images):
        name = os.path.splitext(os.path.basename(low))[0]
        img = cv2.imread(low)
        ref = cv2.imread(high)

        result = apply_filter(
            img,
            mean_params['alpha'], mean_params['beta'],
            mean_params['h'], mean_params['hColor'],
            int(mean_params['template']), int(mean_params['search'])
        )

        cv2.imwrite(os.path.join(out_folder, f"{name}.png"), result)

        psnr, ssim_val, brisque, niqe = apply_metrics(result, ref)
        metrics_list.append({
            "psnr": psnr,
            "ssim": ssim_val,
            "brisque": brisque,
            "niqe": niqe
        })

    mean_metrics = {}
    metric_names = metrics_list[0].keys()

    for metric_name in metric_names:
        values = [m[metric_name] for m in metrics_list]
        mean_metrics[metric_name] = np.mean(values)

    os.makedirs(os.path.dirname(metrics_csv), exist_ok=True)
    pd.DataFrame([mean_metrics]).to_csv(metrics_csv, index=False)
    print(f"[INFO] Average test metrics saved at: {metrics_csv}")

    return mean_metrics


In [13]:
def main():

    train_low, train_high = "LOL_Dataset/our485/low", "LOL_Dataset/our485/high"
    test_low, test_high = "LOL_Dataset/eval15/low", "LOL_Dataset/eval15/high"

    out_test = "results/results_eval15"
    param_csv = "results/mean_train_params.csv"
    metrics_csv = "results/eval15_metrics.csv"

    #Load or compute mean parameters
    if os.path.exists(param_csv):
        print(f"[INFO] Using previously saved parameters from '{param_csv}'")
        mean_params = pd.read_csv(param_csv).iloc[0].to_dict()
    else:
        print("[INFO] Running optimization phase...")
        low_train = load_images_from_folder(train_low)
        high_train = load_images_from_folder(train_high)

        space = [
            Real(1.0, 5.0, name='alpha'),
            Real(0.0, 50.0, name='beta'),
            Real(5.0, 30.0, name='h'),
            Real(5.0, 20.0, name='hColor'),
            Integer(3, 9, name='template'),
            Integer(3, 9, name='search')
        ]
        results = optimization_phase(low_train, high_train, space)
        mean_params = save_information(results, param_csv)

    #Uses the mean parameters obtained from the images in the our485 folder on the images from eval15
    low_test = load_images_from_folder(test_low)
    high_test = load_images_from_folder(test_high)

    mean_metrics = evaluation_phase(low_test, high_test, mean_params, out_test, metrics_csv)

    """
    print("\n=== TEST METRIC AVERAGES ===")
    print(f"PSNR={mean_metrics['psnr']:.3f} | SSIM={mean_metrics['ssim']:.3f} | "
          f"BRISQUE={mean_metrics['brisque']:.3f} | NIQE={mean_metrics['niqe']:.3f}") """
    print("\n[INFO] Pipeline complete.")


In [14]:
if __name__ == "__main__":
    main()

[INFO] Using previously saved parameters from 'results/mean_train_params.csv'
[INFO] Average test metrics saved at: results/eval15_metrics.csv

[INFO] Pipeline complete.
